# D1_04 네 가지 파일을 분석 테이블 하나로

> **오늘 질문:** 흩어진 공장 데이터를 어떻게 하나로 모을까?

| 파일 | 형식 | 내용 |
|---|---|---|
| `equipment_master.json` | JSON (중첩) | 설비 10대의 라인·제조사·설치연도 |
| `production_log.csv` | CSV | 생산 기록 (일시, 설비, 제품, 등급) |
| `sensor_log.parquet` | Parquet | 센서 값 (온도, 회전수, 토크, 공구 마모, 진동) |
| `factory.duckdb` | DB 테이블 | `quality_result`: 고장 여부와 고장 유형 |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
import json
import os

import duckdb
import pandas as pd

DATA = "../../data/bundle/"   # 가상 공장 번들 폴더

## 1. 설비 마스터 (JSON) ✍️

JSON 파일을 열어 읽은 뒤 `pd.json_normalize()`로 표로 펼칩니다.
`sep="_"`를 주면 중첩된 `spec` 아래 값이 `spec_maker`처럼 펼쳐집니다.

> 변수 이름: `equip`

In [ ]:
# 힌트: with open(파일, encoding="utf-8") as f: → json.load(f) → pd.json_normalize(데이터, sep="_")

## 2. 생산 실적 (CSV) ✍️

CSV는 날짜를 글자로 읽습니다. `parse_dates`로 날짜 컬럼을 지정하세요.

> 변수 이름: `prod`

In [ ]:
# 힌트: pd.read_csv(파일, parse_dates=["timestamp"])

## 3. 센서 로그 (Parquet) ✍️

Parquet는 타입을 함께 저장하므로 날짜가 처음부터 날짜로 읽힙니다.

> 변수 이름: `sensor`

In [ ]:
# 힌트: pd.read_parquet(파일)

## 4. 품질 결과 (DuckDB) ✍️

DB 파일에 연결하고 SQL로 테이블을 조회합니다. `.df()`를 붙이면 pandas 표로 바뀝니다.

> 변수 이름: `con`

In [ ]:
# 힌트: duckdb.connect(파일, read_only=True) → con.sql("SQL").df()

## 5. 조인 전에 키 확인 📋

생산 실적과 센서 로그는 **설비ID + 일시**로 붙입니다. 두 표의 `timestamp` 타입이 같아야 합니다.

In [ ]:
print(prod["timestamp"].dtype, sensor["timestamp"].dtype)

## 6. DuckDB로 한 번에 조인 ✍️

DuckDB는 DB 안의 테이블(`quality_result`)과 파이썬 변수에 담긴 표(`prod`, `sensor`, `equip`)를 함께 SQL로 조인할 수 있습니다.

| 붙이는 표 | 키 |
|---|---|
| 센서 로그 `s` | 설비ID + 일시 |
| 품질 결과 `q` | 제품ID |
| 설비 마스터 `e` | 설비ID |

> 변수 이름: `sql`, `df`

In [ ]:
# 힌트: LEFT JOIN 표 별칭 ON 키 = 키, 컬럼 AS 새이름

## 7. 조인 결과 확인 ✍️

조인 후에는 **행 수가 그대로인지** 꼭 확인합니다. 키가 어긋나면 행이 늘거나, 값이 비게 됩니다.

**Q1.** 생산 기록은 10,000건이라고 들었는데 행 수는 몇 개인가요? 왜 그럴까요?

> 답:


## 8. CSV vs Parquet 저장 비교 ✍️

같은 분석 테이블을 두 형식으로 저장해 크기와 읽는 속도를 비교합니다.

In [ ]:
# 힌트: 표.to_csv(파일, index=False), 표.to_parquet(파일, index=False)

In [ ]:
csv_kb = os.path.getsize("analysis_table.csv") / 1024
pq_kb = os.path.getsize("analysis_table.parquet") / 1024
print(f"CSV     {csv_kb:,.0f} KB")
print(f"Parquet {pq_kb:,.0f} KB  (CSV의 {pq_kb / csv_kb:.0%})")

`%timeit`은 같은 코드를 여러 번 실행해 평균 시간을 잽니다.

In [ ]:
# 힌트: %timeit -n 5 코드

## 정리 💬

**Q2.** 매일 쌓이는 센서 로그를 분석용으로 저장한다면 어떤 형식을 고르겠습니까? 이유는?

> 답:
